# Notebook 01 -- Data Overview

**Thesis:** Content Moderation on TikTok and X -- A Comparative Analysis of DSA Statements of Reasons (2025)  
**Author:** Mohsen Zahedi  

> **Note:** This notebook is pre-run. Outputs were produced on the author's machine. The full dataset (~120 GB) lives at `C:\BA_Data\` and is not included in this repository. To reproduce, point `DATA_ROOT` to your local copy of the parquet files.

---

This notebook introduces the dataset. It covers what DSA Statement of Reasons data is, which platforms are studied, how large the dataset is, and what each column represents.

## 1. What is DSA Statement of Reasons Data?

Under **Article 17 of the EU Digital Services Act (DSA)**, Very Large Online Platforms (VLOPs) must submit a **Statement of Reasons (SoR)** to the [DSA Transparency Database](https://transparency.dsa.ec.europa.eu/) each time they take a content moderation decision.

Each SoR record captures:

| Field | What it records |
|---|---|
| `category` | Which DSA violation category the content falls under |
| `content_type` | What kind of content was moderated (video, account, comment, etc.) |
| `automated_detection` | Whether automated tools detected the violation |
| `automated_decision` | Whether the moderation decision was made automatically |
| `decision_visibility` | How the content's visibility was restricted (removed, labelled, etc.) |
| `application_date` | When the decision was applied |
| `territorial_scope` | Which countries the restriction applies to |

This thesis analyzes SoR data for **TikTok** and **X (formerly Twitter)** for the calendar year **2025**, restricted to decisions covering **Germany (DE)**.

In [1]:
from pathlib import Path

import polars as pl

DATA_ROOT = Path(r"C:\BA_Data")
TIKTOK_FILE = DATA_ROOT / "tiktok_de_2025.parquet"
X_FILE = DATA_ROOT / "x_de_2025.parquet"

## 2. Dataset Scale

The two platforms differ dramatically in reporting volume.

In [2]:
tiktok_rows = pl.scan_parquet(str(TIKTOK_FILE)).select(pl.len()).collect().item()
x_rows = pl.scan_parquet(str(X_FILE)).select(pl.len()).collect().item()

print(f"Platform   {'Rows':>15}")
print("-" * 30)
print(f"TikTok     {tiktok_rows:>15,}")
print(f"X          {x_rows:>15,}")
print("-" * 30)
print(f"Scale ratio TikTok:X = {tiktok_rows / x_rows:,.0f}:1")

Platform              Rows
------------------------------
TikTok         997,688,267
X                  183,321
------------------------------
Scale ratio TikTok:X = 5,442:1


## 3. Schema

Both platforms share the same 10 columns. These are the nine DSA in-scope fields plus `api_version`, which distinguishes submissions made under DSA API v1 (before July 1, 2025) from v2 (from July 1, 2025 onwards). The v1/v2 distinction matters because the `category` vocabulary changed -- this is addressed in Notebook 03.

In [3]:
schema = pl.read_parquet_schema(str(TIKTOK_FILE))
print(f"{'Column':<30} {'Type'}")
print("-" * 50)
for col, dtype in schema.items():
    print(f"{col:<30} {dtype}")

Column                         Type
--------------------------------------------------
uuid                           String
category                       String
content_type                   String
automated_detection            String
automated_decision             String
territorial_scope              String
application_date               Datetime(time_unit='ms', time_zone=None)
platform_name                  String
decision_visibility            String
api_version                    String


## 4. Column Descriptions

| Column | Type | Description |
|---|---|---|
| `uuid` | String | Unique identifier per SoR record |
| `category` | String | DSA violation category as submitted by the platform (raw, not harmonized) |
| `content_type` | String | Type of content moderated: `VIDEO`, `ACCOUNT`, `POST`, etc. |
| `automated_detection` | String | Whether automated tools flagged the content (`true` / `false`) |
| `automated_decision` | String | Whether the final moderation decision was made automatically |
| `territorial_scope` | String | Country codes where the restriction applies (may be multi-country, e.g. `DE,FR,NL`) |
| `application_date` | Datetime | Date the moderation decision was applied (no timezone in raw data) |
| `platform_name` | String | Name of the platform filing the SoR |
| `decision_visibility` | String | How content visibility was restricted (e.g. `REMOVED_CONTENT`, `LABELLED`) |
| `api_version` | String | DSA API version used to submit the SoR (`v1` or `v2`) |

## 5. Sample Rows

### TikTok

In [4]:
pl.scan_parquet(str(TIKTOK_FILE)).head(5).collect()

uuid,category,content_type,automated_detection,automated_decision,territorial_scope,application_date,platform_name,decision_visibility,api_version
str,str,str,str,str,str,datetime[ms],str,str,str
"""0b3c3647-9806-419c-bd99-de66b6…","""STATEMENT_CATEGORY_SCOPE_OF_PL…","""[""CONTENT_TYPE_VIDEO""]""","""Yes""","""AUTOMATED_DECISION_FULLY""","""[""AT"",""BE"",""BG"",""CY"",""CZ"",""DE""…",2025-01-01 00:00:00,"""TikTok""","""[""DECISION_VISIBILITY_OTHER""]""","""v1"""
"""8415527b-855e-437b-a698-4eef9b…","""STATEMENT_CATEGORY_SCOPE_OF_PL…","""[""CONTENT_TYPE_VIDEO""]""","""Yes""","""AUTOMATED_DECISION_FULLY""","""[""AT"",""BE"",""BG"",""CY"",""CZ"",""DE""…",2025-01-01 00:00:00,"""TikTok""","""[""DECISION_VISIBILITY_OTHER""]""","""v1"""
"""27ebdca3-085f-47d3-91ad-79e35b…","""STATEMENT_CATEGORY_ILLEGAL_OR_…","""[""CONTENT_TYPE_TEXT""]""","""Yes""","""AUTOMATED_DECISION_FULLY""","""[""AT"",""BE"",""BG"",""CY"",""CZ"",""DE""…",2025-01-01 00:00:00,"""TikTok""","""[""DECISION_VISIBILITY_CONTENT_…","""v1"""
"""73aa4ad5-3480-4597-b1aa-c3beb2…","""STATEMENT_CATEGORY_SCOPE_OF_PL…","""[""CONTENT_TYPE_VIDEO""]""","""Yes""","""AUTOMATED_DECISION_FULLY""","""[""AT"",""BE"",""BG"",""CY"",""CZ"",""DE""…",2025-01-01 00:00:00,"""TikTok""","""[""DECISION_VISIBILITY_OTHER""]""","""v1"""
"""f277ba47-b2ab-4e97-b5a8-ec7e41…","""STATEMENT_CATEGORY_SCOPE_OF_PL…","""[""CONTENT_TYPE_VIDEO""]""","""Yes""","""AUTOMATED_DECISION_FULLY""","""[""AT"",""BE"",""BG"",""CY"",""CZ"",""DE""…",2025-01-01 00:00:00,"""TikTok""","""[""DECISION_VISIBILITY_OTHER""]""","""v1"""


### X

In [5]:
pl.scan_parquet(str(X_FILE)).head(5).collect()

uuid,category,content_type,automated_detection,automated_decision,territorial_scope,application_date,platform_name,decision_visibility,api_version
str,str,str,str,str,str,datetime[ms],str,str,str
"""457e68f1-ed59-470e-9a6b-9902d2…","""STATEMENT_CATEGORY_SCAMS_AND_F…","""[""CONTENT_TYPE_SYNTHETIC_MEDIA…","""No""","""AUTOMATED_DECISION_NOT_AUTOMAT…","""[""DE""]""",2025-01-01 00:00:00,"""X""","""[""DECISION_VISIBILITY_OTHER""]""","""v1"""
"""a07aa67e-adbe-4f3e-ab4b-c485b4…","""STATEMENT_CATEGORY_SCAMS_AND_F…","""[""CONTENT_TYPE_SYNTHETIC_MEDIA…","""No""","""AUTOMATED_DECISION_NOT_AUTOMAT…","""[""DE""]""",2025-01-01 00:00:00,"""X""","""[""DECISION_VISIBILITY_OTHER""]""","""v1"""
"""604e6f72-6a67-43ab-8f77-003c66…","""STATEMENT_CATEGORY_SCAMS_AND_F…","""[""CONTENT_TYPE_SYNTHETIC_MEDIA…","""No""","""AUTOMATED_DECISION_NOT_AUTOMAT…","""[""DE""]""",2025-01-01 00:00:00,"""X""","""[""DECISION_VISIBILITY_OTHER""]""","""v1"""
"""9487ad6c-e72d-493d-b25f-4f9826…","""STATEMENT_CATEGORY_PORNOGRAPHY…","""[""CONTENT_TYPE_SYNTHETIC_MEDIA…","""No""","""AUTOMATED_DECISION_NOT_AUTOMAT…","""[""DE""]""",2025-01-01 00:00:00,"""X""","""[""DECISION_VISIBILITY_OTHER""]""","""v1"""
"""3eadd52f-deef-45c9-9f3d-d419a2…","""STATEMENT_CATEGORY_UNSAFE_AND_…","""[""CONTENT_TYPE_SYNTHETIC_MEDIA…","""No""","""AUTOMATED_DECISION_NOT_AUTOMAT…","""[""DE""]""",2025-01-01 00:00:00,"""X""","""[""DECISION_VISIBILITY_OTHER""]""","""v1"""


## 6. Key Observations

- **Scale:** TikTok filed roughly 5,400x more SoRs than X for Germany in 2025. This is a central empirical finding of the thesis.
- **Categories:** The `category` field uses different label sets in v1 and v2 submissions. Raw labels are therefore not directly comparable across the two API versions -- see Notebook 03 for the harmonization approach.
- **Territorial scope:** TikTok entries frequently list multiple countries (e.g. `DE,FR,IT`), reflecting EEA-wide enforcement. X entries use single-country codes only, reflecting country-specific enforcement.
- **All columns are categorical or temporal.** There are no numeric measurement columns, so variance is not a meaningful metric for this dataset.